# Reference
- https://github.com/MyoHub/myosuite/blob/main/docs/source/tutorials/2_Load_policy.ipynb

# Issues

- 20260326, Some errors happen during loading mujoco, mujoco.py
    - pip install "cython<3.0"
    - sudo apt install libglew-dev libosmesa6-dev libgl1-mesa-glx libglfw3 patchelf
    - setting library paths
        ```python
        mujoco_bin = "/home/seojin/mujoco210/bin"
        nvidia_path = "/usr/lib/nvidia"
        current_ld = os.environ.get("LD_LIBRARY_PATH", "")
        os.environ["LD_LIBRARY_PATH"] = f"{current_ld}:{mujoco_bin}:{nvidia_path}"
        
        # 1. Set the path so mujoco_py knows where to find the 'headers' for compilation
        os.environ["MUJOCO_PY_MUJOCO_PATH"] = "/home/seojin/mujoco210"

# Common Libraries

In [2]:
import os
os.environ["MUJOCO_GL"] = "egl"

import sys
import pickle
import skvideo.io
import numpy as np
from myosuite.utils import gym

# Custom Libraries

In [3]:
sys.path.append("/home/seojin/Seojin_commonTool/Module")
from jupyter_util import show_video

ModuleNotFoundError: No module named 'jupyter_util'

# Configuration

In [ ]:
mujoco_bin = "/home/seojin/mujoco210/bin"
nvidia_path = "/usr/lib/nvidia"
current_ld = os.environ.get("LD_LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = f"{current_ld}:{mujoco_bin}:{nvidia_path}"
os.environ["MUJOCO_PY_MUJOCO_PATH"] = "/home/seojin/mujoco210"
import mjrl
import myosuite

# Environment

In [4]:
env = gym.make('myoElbowPose1D6MExoRandom-v0')

    MyoSuite: A contact-rich simulation suite for musculoskeletal motor control
        Vittorio Caggiano, Huawei Wang, Guillaume Durandau, Massimo Sartori, Vikash Kumar
        L4DC-2019 | https://sites.google.com/view/myosuite
    


# Policy

In [5]:
dir_path = "/home/seojin/myosuite/myosuite/agents/baslines_NPG"
policy_path = os.path.join(dir_path, "myoElbowPose1D6MExoRandom-v0/2022-02-26_21-16-27/36_env=myoElbowPose1D6MExoRandom-v0,seed=1/iterations/best_policy.pickle")
with open(policy_path, 'rb') as f:
    pi = pickle.load(f)

# Simulation

This is a "Position Tracking" test. You are forcing the elbow to move to specific angles in a sequence to see how well your trained policy (pi) can reach and hold those positions.

Every 40 steps, the target jumps to a new angle from your AngleSequence. The policy has 40 frames to try and move the arm to that new position. If the policy is well-trained, you will see the elbow smoothly chasing the targets in the resulting video.

- unwrapped
    Normally, a Gym environment only lets you use three main functions: reset(), step(), and render(). It doesn't want you touching the "internals."

    However, in your specific script, you are doing things a normal user shouldn't do during training, such as:
    - Manually changing the target angle: env.unwrapped.target_jnt_value = ...
    - Changing the task type on the fly: env.unwrapped.target_type = 'fixed'
    - Modifying the physics (weight): env.unwrapped.weight_range = (0,0)
    
    The standard env object doesn't have these attributes. They belong specifically to the MyoSuite Elbow Environment. By calling .unwrapped, you "reach past" the Gym safety layers to grab those specific variables.

In [ ]:
# This is a list of target elbow angles in degrees
AngleSequence = [60, 30, 30, 60, 80, 80, 60, 30, 80, 30, 80, 60]

In [6]:
env.reset()
frames = []
for ep in range(len(AngleSequence)):
    print("Ep {} of {} testing angle {}".format(ep, len(AngleSequence), AngleSequence[ep]))
    env.unwrapped.target_jnt_value = [np.deg2rad(AngleSequence[int(ep)])]
    env.unwrapped.target_type = 'fixed'
    env.unwrapped.weight_range=(0,0)
    env.unwrapped.update_target()
    for _ in range(40):
        frame = env.unwrapped.sim.renderer.render_offscreen(width=400,
                                                            height=400,
                                                            camera_id=0)
        frames.append(frame)
        o = env.unwrapped.get_obs()
        a = pi.get_action(o)[0]
        next_o, r, done, *_, ifo = env.step(a) # take an action based on the current observation
env.close()

Ep 0 of 12 testing angle 60
Ep 1 of 12 testing angle 30
Ep 2 of 12 testing angle 30
Ep 3 of 12 testing angle 60
Ep 4 of 12 testing angle 80
Ep 5 of 12 testing angle 80
Ep 6 of 12 testing angle 60
Ep 7 of 12 testing angle 30
Ep 8 of 12 testing angle 80
Ep 9 of 12 testing angle 30
Ep 10 of 12 testing angle 80
Ep 11 of 12 testing angle 60


# Save video

In [7]:
video_dir_path = os.path.join("/mnt/ext1/seojin/temp")
video_path = os.path.join(video_dir_path, "temp.mp4")
skvideo.io.vwrite(video_path, np.asarray(frames),outputdict={"-pix_fmt": "yuv420p"})
print(f"save: {video_path}")

save: /mnt/ext1/seojin/temp/temp.mp4


# Show video

In [8]:
show_video(video_path)